## Part 0: Import Required Packages

In [ ]:
# Import required libraries
import pandas as pd
import matplotlib.pyplot as plt
import os
import scipy.stats as stat
import time
import csv
import seaborn as sns

## Part 1: Data Preprocessing

### 1.1 Sample Data Preprocessing

In [ ]:
# Define function for preprocessing sample data
def samples_data_preprocessing(df_file):
    df_file = df_file.groupby(df_file.index).mean()  # Remove duplicates by averaging
    df_file = df_file.dropna(axis=0, how='all')       # Drop rows where all values are NaN
    df_file = df_file.loc[(df_file != 0).any(1)]      # Drop rows where all values are 0
    return df_file

In [ ]:
# Load and preprocess reference samples
reference_samples_file = pd.read_csv("data/raw_data_57_reference_samples.csv", index_col="GeneSymbol")
reference_samples_file = samples_data_preprocessing(reference_samples_file)
reference_samples_file

In [ ]:
# Load and preprocess experimental samples
n_samples_file = pd.read_csv("data/raw_data_8_samples_data.csv", index_col="GeneSymbol")
n_samples_file = samples_data_preprocessing(n_samples_file)
n_samples_file

In [ ]:
# Merge the preprocessed reference and experimental samples
merge_initial_file = pd.merge(reference_samples_file, n_samples_file, on='GeneSymbol')
merge_initial_file

### 1.2 Preprocessing Protein-Protein Interaction (PPI) Data

In [ ]:
# Define function for preprocessing PPI data
def data_preprocessing_with_PPI(df_file):
    df_info = pd.read_csv("data/string-db-9606.protein/9606.protein.info.v11.5.txt", sep="\t")  # Load protein info
    df_file = df_file.groupby(df_file.index).mean()    # Remove duplicates by averaging
    df_file = df_file.dropna(axis=0, how='all')        # Drop rows where all values are NaN
    df_file = df_file.loc[(df_file != 0).any(1)]       # Drop rows where all values are 0
    df_file = df_file.reset_index().merge(df_info.iloc[:, :2], how='inner', left_on='GeneSymbol', right_on='preferred_name')
    df_id2name = df_file.iloc[:, -2:]                  # Extract columns for gene ID mapping
    df_file = df_file.drop(df_id2name.columns, axis=1).set_index('GeneSymbol')  # Remove ID columns from main data
    return df_id2name, df_file

# Apply preprocessing to merged data
id2name, data = data_preprocessing_with_PPI(merge_initial_file)

In [ ]:
id2name

In [ ]:
# Split data into reference and experimental samples
ref_file = data.iloc[:, :57]    # Reference samples
per_file = data.iloc[:, 57:]    # Experimental samples

In [ ]:
ref_file.head()

In [ ]:
per_file.head()

In [ ]:
# Save processed data
res_dir01 = 'results/01.ssn_input'
os.makedirs(res_dir01, exist_ok=True)
ref_file.to_csv(f'{res_dir01}/ref_file_57_reference_samples.txt', sep='\t')  # Save reference samples
per_file.to_csv(f'{res_dir01}/per_file_8_samples_data.txt', sep='\t')         # Save experimental samples

## Part 2: Construct Background Network File

In [ ]:
# Define function to construct background network file based on PPI data
def background_network_file(df_id2name):
    df_link = pd.read_csv("data/string-db-9606.protein/9606.protein.links.v11.5.txt", sep=" ")
    df_link = df_link.merge(df_id2name, left_on='protein1', right_on='#string_protein_id')
    df_link = df_link.merge(df_id2name, left_on='protein2', right_on='#string_protein_id')
    df_link = df_link[['preferred_name_x', 'preferred_name_y', 'combined_score']]
    df_link = df_link[df_link['combined_score'] >= 700]  # Filter edges with a combined score >= 700
    return df_link

# Create and save background network file
bg_net_file = background_network_file(id2name)
bg_net_file.iloc[:, :2].to_csv(f'{res_dir01}/background.txt', sep='\t', index=None, header=None)

In [ ]:
bg_net_file.head()

## Part 3: SSN Construction

### 3.1 SSN Score Calculation Function

In [ ]:
# Define function to calculate SSN score
def ssn_score(delta, pcc, nn):
    pcc = min(max(pcc, -0.99999999), 0.99999999)  # Avoid exact 1 or -1 for Pearson correlation
    z = delta / ((1 - pcc**2) / (nn - 1))
    return z

### 3.2 Construct Single SSN Network

In [ ]:
# Define function to construct SSN network for each sample
def construct_single_network(f_ref, f_per, f_bg, dir_fold, param_pvalue):
    begin = time.asctime()
    print("Start time:", begin)
    
    reference_data, sample_data = {}, {}
    reference_file, sample_file, background_file = f_ref, f_per, f_bg

    # Load reference samples
    with open(reference_file) as f:
        header = f.readline().split()
        for line in f:
            parts = line.split()
            reference_data[parts[0]] = list(map(float, parts[1:]))

    # Load sample data
    with open(sample_file) as f:
        header = f.readline().split()
        for line in f:
            parts = line.split()
            sample_data[parts[0]] = list(map(float, parts[1:]))

    os.makedirs(dir_fold, exist_ok=True)

    # Process each sample
    for i, sample_name in enumerate(header[1:], start=1):
        print(f"Processing sample {sample_name} at {time.asctime()}")
        output_path = os.path.join(dir_fold, f"ssn_{sample_name}.txt")
        
        with open(output_path, "w") as fw:
            fw.write("Gene1\tGene2\tdeltaPCC\tp-value\n")
            with open(background_file) as f:
                for line in f:
                    gene1, gene2 = line.split()
                    r1 = stat.pearsonr(reference_data[gene1], reference_data[gene2])[0]
                    r2 = stat.pearsonr(reference_data[gene1] + [sample_data[gene1][i-1]], 
                                       reference_data[gene2] + [sample_data[gene2][i-1]])[0]
                    delta_pcc = r2 - r1
                    z = ssn_score(delta_pcc, r1, len(reference_data[gene1]))
                    p_value = 1 - stat.norm.cdf(abs(z))
                    
                    if p_value < param_pvalue:
                        fw.write(f"{gene1}\t{gene2}\t{delta_pcc}\t{p_value}\n")

        print(f"Finished processing {sample_name} at {time.asctime()}")
    
    print("End time:", time.asctime())

In [ ]:
# Define output directory for SSN networks
res_dir02 = 'results/02.ssn_output'
os.makedirs(res_dir02, exist_ok=True)

# Run SSN construction with given parameters
construct_single_network(
    f_ref=f'{res_dir01}/ref_file_57_reference_samples.txt', 
    f_per=f'{res_dir01}/per_file_8_samples_data.txt',
    f_bg=f'{res_dir01}/background.txt',
    dir_fold=res_dir02,
    param_pvalue=0.01
)

## Part 4: SSN Output Files and Process

In [ ]:
# Define function to read and process SSN output files
def get_read_file(file_dir, file_path):
    file_name = file_path.split('.')[0]
    print(f"Processing file: {file_name}")
    df = pd.read_csv(os.path.join(file_dir, file_path), sep='\t')
    df['edge'] = df['Gene1'] + '+' + df['Gene2']
    df = df[['edge', 'deltaPCC']].rename(columns={'deltaPCC': file_name})
    return df

# Define function to merge multiple SSN result files
def merge_res_file(files_dir, output_file):
    files_list = os.listdir(files_dir)
    df = get_read_file(files_dir, files_list[0])
    
    for i, filepath in enumerate(files_list[1:], start=1):
        df_temp = get_read_file(files_dir, filepath)
        df = pd.merge(df, df_temp, on='edge')
        print(f"Merged {i + 1} files")
    
    df.to_csv(output_file, index=False, header=True)
    print("Merging completed.")
    return df

# Define directory for merged SSN output
res_dir03 = 'results/03.ssn_merge'
os.makedirs(res_dir03, exist_ok=True)

# Perform merging of SSN output files
res_merge = merge_res_file(files_dir=res_dir02, output_file=f'{res_dir03}/ssn_res_merge.csv')
res_merge = res_merge.set_index('edge')

In [ ]:
res_merge

## Part 5: T-Test Analysis

### 5.1 Separate Data for T-Test and Define T-Test Function

In [ ]:
# Load merged SSN results
df_ssn_res = pd.read_csv(f'{res_dir03}/ssn_res_merge.csv').set_index('edge')

# Separate data into two groups for T-test analysis
res_ssn_type1 = df_ssn_res[['ssn_GSM1498849', 'ssn_GSM1498850', 'ssn_GSM1498851', 'ssn_GSM1498852']]
res_ssn_type2 = df_ssn_res[['ssn_GSM1498853', 'ssn_GSM1498854', 'ssn_GSM1498855', 'ssn_GSM1498856']]

# Define T-test function
def t_test(a, b):
    lev_p = stat.levene(a, b)[1]    # Levene test for equal variances
    equal_var = lev_p > 0.05        # True if variances are equal
    res = stat.ttest_ind(a, b, equal_var=equal_var)
    return lev_p, int(equal_var), res[1], int(res[1] < 0.05)

# Function to get T-test results
def get_t_test_result(df_type1, df_type2, result_path):
    df_former_tr = df_type1.T
    df_latter_tr = df_type2.T

    with open(result_path, 'w', encoding='utf-8', newline="") as f:
        csv_writer = csv.writer(f)
        csv_writer.writerow(["edge", 'variance', 'bool_variance', 'p_value', 'bool_significant'])
        
        for i in df_former_tr.columns:
            res = t_test(df_former_tr[i].tolist(), df_latter_tr[i].tolist())
            csv_writer.writerow([i, *res])

# Define directory for T-test results
res_dir04 = 'results/04.ssn_ttest'
os.makedirs(res_dir04, exist_ok=True)

# Run T-test and save results
get_t_test_result(res_ssn_type1, res_ssn_type2, f'{res_dir04}/ttest_result_bool.csv')

In [ ]:
res_ssn_type1.head()

In [ ]:
res_ssn_type2.head()

### 5.2 Process T-Test Results

In [ ]:
# Function to filter and process T-test results
def ttest_res_processing(csv_path):
    df = pd.read_csv(csv_path)
    df = df[(df['bool_variance'] != 0) & (df['bool_significant'] != 0)]
    df = df[['edge', 'p_value']]
    return df

# Process T-test results and save
df_ttest_res = ttest_res_processing(f'{res_dir04}/ttest_result_bool.csv')
df_ttest_res.to_csv(f'{res_dir04}/ttest_result_final.csv', index=None)

In [ ]:
df_ttest_res

## Part 6: Statistical Analysis and Gene Networks

### Notes

**Gene Network Construction and Visualization**  
   Gene network construction and visualization were performed using **Cytoscape**.  
   Please refer to the directory: `results/06.res_candidate_gene_network/SSN4PaCDM_XXXX.cys`.

### 6.1 Statistical Analysis and Candidate Gene Filtering

In [ ]:
# Load T-test results
res_edges = pd.read_csv(f'{res_dir04}/ttest_result_final.csv')
res_edges

In [ ]:
# Process "edge" column: sort genes alphabetically and remove duplicates
res_edges['edge'] = res_edges['edge'].apply(lambda x: '+'.join(sorted(x.split('+'))))
res_edges = res_edges.drop_duplicates()

# Sort by p-value
res_edges = res_edges.sort_values(by='p_value', ascending=True)

# Split edges into Gene1 and Gene2 for further analysis
res_edges["Gene1"] = res_edges["edge"].apply(lambda x: x.split("+")[0])
res_edges["Gene2"] = res_edges["edge"].apply(lambda x: x.split("+")[1])

# Save processed edges
res_dir05 = 'results/05.ssn_different_edges'
os.makedirs(res_dir05, exist_ok=True)
res_edges.to_csv(f'{res_dir05}/01.ssn_all_different_edges.csv', index=None)

In [ ]:
res_edges

### 6.2 Top Candidate Genes and Degree Analysis

In [ ]:
# Filter edges with p-value < 0.0001
filtered_edges = res_edges[res_edges['p_value'] < 0.0001]

# Save top candidate network edges
res_dir06 = 'results/06.res_candidate_gene_network'
os.makedirs(res_dir06, exist_ok=True)
filtered_edges[["Gene1", "Gene2", "p_value"]].to_csv(
    f'{res_dir06}/01.SSN4PaCDM_function_gene_network(975_different_edges_p<0.0001).csv', index=None
)

**01.screen_SSN4PaCDM_function_gene_nework(975_different_edges_p<0.0001)**

![Cytoscape](./figures/01.screen_SSN4PaCDM_function_gene_nework_975_different_edges_0001.png)

In [ ]:
# Identify top candidate genes based on degree
degree_dict = {}
for _, row in filtered_edges.iterrows():
    for gene in [row['Gene1'], row['Gene2']]:
        degree_dict[gene] = degree_dict.get(gene, 0) + 1

# Sort genes by degree and select top 10
sorted_degrees = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)
top_10_genes = [gene for gene, _ in sorted_degrees[:10]]

# Save top candidate genes
with open(f'{res_dir05}/02.SSN4PaCDM_top10_candidate_genes.csv', 'w') as f:
    for gene in top_10_genes:
        f.write(f"{gene}\n")

# Filter network by top 10 genes
function_gene_network_top975 = filtered_edges[
    filtered_edges['Gene1'].isin(top_10_genes) | filtered_edges['Gene2'].isin(top_10_genes)
]
function_gene_network_top975[["Gene1", "Gene2", "p_value"]].to_csv(
    f'{res_dir06}/02.SSN4PaCDM_function_gene_network(top10_candidate_genes).csv', index=None
)

**02.screen_SSN4PaCDM_function_gene_network(top10_candidate_genes)**

![Cytoscape](./figures/02.screen_SSN4PaCDM_function_gene_network_top10_candidate_genes.png)

### 6.3 Degree Statistics and Visualization

In [ ]:
# Step 1: Prepare unique gene list
gene1_list = res_edges['Gene1'].unique()
gene2_list = res_edges['Gene2'].unique()
all_genes = set(gene1_list).union(gene2_list)
print(f"Total unique genes: {len(all_genes)}")

# Step 2: Initialize degree dictionary
degree_dict = {gene: [] for gene in all_genes}

# Step 3: Calculate degree statistics for top N edges
for i in range(10, 976, 1):
    # print(i)
    edges_top_i = res_edges.head(i)     # 100, 150, 200, 500, 1000, 1500, 2000, 3000, 4000, 5000, all
    degree_dict_i = {}

    for _, row in edges_top_i.iterrows():
        gene1 = row['Gene1']
        gene2 = row['Gene2']
        
        if gene1 in degree_dict_i:
            degree_dict_i[gene1] += 1
        else:
            degree_dict_i[gene1] = 1
        
        if gene2 in degree_dict_i:
            degree_dict_i[gene2] += 1
        else:
            degree_dict_i[gene2] = 1

    sorted_degrees_i = sorted(degree_dict_i.items(), key=lambda x: x[1], reverse=True)
    top_10_genes_list_i = [gene for gene, _ in sorted_degrees_i[:10]]
    # top_10_genes_list_i = [gene for gene, _ in sorted_degrees_i]
    # print(top_10_genes_list_i)

    for gene, _ in sorted_degrees_i[:10]:
    # for gene, _ in sorted_degrees_i:
        degree_dict[gene].append(_)
    for k in degree_dict:
        if k in top_10_genes_list_i:
            continue
        else:
            degree_dict[k].append(0)

# Convert degree dictionary to DataFrame
df_degree_dict = pd.DataFrame(degree_dict)
df_degree_dict

In [ ]:
# Filter for top 10 candidate genes
df_degree_dict = df_degree_dict[['WDR31', 'MRPL11', 'RAF1', 'DDX20', 'PSMC1', 'KIF14', 'PLK1', 'RPS24', 'NEK2', 'PRKAR1B']]
df_degree_dict.to_csv(res_dir05+'/03.SSN4PaCDM_top10_candidate_genes_degree_statistics.csv', index=None)
df_degree_dict

In [ ]:
# Step 4: Calculate IQR and Median for degree statistics
iqr_values = df_degree_dict.quantile(0.75) - df_degree_dict.quantile(0.25)
median_values = df_degree_dict.median()

# Sort columns by IQR
sorted_columns = iqr_values.sort_values().index

# Step 5: Plot boxplot for degree statistics
fig, ax = plt.subplots(figsize=(8, 4))

# Boxplot with sorted columns
ax.boxplot(df_degree_dict[sorted_columns].values)

# Define box colors
box_colors = ['#8cc2de', '#8cc2de', '#8cc2de', '#f3b11b', '#f3b11b', '#8cc2de', '#8cc2de', '#8cc2de', '#8cc2de', '#8cc2de']
boxplot = ax.boxplot(df_degree_dict[sorted_columns].values, patch_artist=True)
for i, box in enumerate(boxplot['boxes']):
    box.set(facecolor=box_colors[i % len(box_colors)])

# Set median line color
median_color = 'red'
for median in boxplot['medians']:
    median.set(color=median_color)

# Customize axis labels and ticks
ax.set_xticks(range(1, len(sorted_columns) + 1))
ax.set_xticklabels(sorted_columns, rotation=45, ha='right')
ax.set_xlabel('Candidate genes')
ax.set_ylabel('Degrees values')

# Add horizontal line at 0
plt.axhline(0, color='red', linestyle=':')

# Save plot as PDF
plt.savefig(f'{res_dir05}/04.SSN4PaCDM_top10_candidate_genes_IQR_degrees_Boxplot.pdf', bbox_inches="tight", dpi=300)
plt.show()


In [ ]:
# Step 6: Save IQR and Median values as a table
IQR_df = pd.DataFrame({
    'Gene': sorted_columns,
    'IQR': iqr_values,
    'Median': median_values
}).sort_values(by='IQR', ascending=True)
# Save to CSV
IQR_df.to_csv(f'{res_dir05}/05.SSN4PaCDM_top10_candidate_genes_IQR_degrees_Table.csv', index=None)
IQR_df

### 6.4: Analysis for Specific Genes (NEK2 and PLK1 Networks)

In [ ]:
# Analyze specific gene networks for NEK2 and PLK1
ssn_all_edges = pd.read_csv(f'{res_dir05}/01.ssn_all_different_edges.csv')

In [ ]:
# Extract edges containing PLK1
PLK1_network = ssn_all_edges[ssn_all_edges['edge'].str.contains('PLK1')]
PLK1_network.to_csv(f'{res_dir06}/03.SSN4PaCDM_PLK1_network.csv', index=None)
PLK1_network

**03.screen_SSN4PaCDM_PLK1_network**

![Cytoscape](./figures/03.screen_SSN4PaCDM_PLK1_network.png)

In [ ]:
# Extract edges containing NEK2
NEK2_network = ssn_all_edges[ssn_all_edges['edge'].str.contains('NEK2')]
NEK2_network.to_csv(f'{res_dir06}/04.SSN4PaCDM_NEK2_network.csv', index=None)
NEK2_network

**04.screen_SSN4PaCDM_NEK2_network**

![Cytoscape](./figures/04.screen_SSN4PaCDM_NEK2_network.png)

### 6.5: Select Target Gene List for KEGG pathway analysis

In [ ]:
diff_edges = pd.read_csv(res_dir06+'/01.SSN4PaCDM_function_gene_nework(975_different_edges_p<0.0001).csv')
diff_edges

In [ ]:
# Target genes
target_genes = ["WDR31", "MRPL11", "RAF1", "DDX20", "PSMC1", "KIF14", "PLK1", "RPS24", "NEK2", "PRKAR1B"]

# Find genes connected to target genes in Gene1 and Gene2
connected_genes_1 = diff_edges[diff_edges['Gene1'].isin(target_genes)]['Gene2']
connected_genes_2 = diff_edges[diff_edges['Gene2'].isin(target_genes)]['Gene1']

# Combine all connected genes and remove duplicates
connected_genes = pd.concat([connected_genes_1, connected_genes_2]).drop_duplicates().tolist()

# Merge target genes with connected genes and remove duplicates
kegg_gene_list = list(set(target_genes + connected_genes))

# Save all genes as a KEGG gene list
res_dir07 = 'results/07.res_kegg_pathway_analysis'
os.makedirs(res_dir07, exist_ok=True)
with open(f'{res_dir07}/01.SSN4PaCDM_kegg_gene_list.txt', 'w') as f:
    for gene in kegg_gene_list:
        f.write(f"{gene}\n")

In [ ]:
print(len(target_genes))
print(len(connected_genes))
print(len(kegg_gene_list))

In [ ]:
kegg_gene_list

## Part 7: KEGG pathway Analysis

### Notes

**Functional Gene Network Enrichment Pathway Analysis**  
   Functional gene network pathway enrichment analysis was conducted using **R**.  
   Please refer to the directory: `results/07.res_kegg_pathway_analysis/SSN4PaCDM_KEGG_pathway_analysis.R` for the R script and results.


**KEGG pathway Analysis**

insulin_kegg_pathway             |  kegg_scaled
:-------------------------:|:-------------------------:
![Cytoscape](./results/07.res_kegg_pathway_analysis/03.SSN4PaCDM_insulin_kegg_pathway.png)|![Cytoscape](./results/07.res_kegg_pathway_analysis/05.SSN4PaCDM_kegg_scaled.png)